In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef
from lightgbm import LGBMClassifier

RANDOM_STATE_1 = 42
RANDOM_STATE_2 = 2024
N_SPLITS = 5


def mcc_optimal_threshold(y_true, proba, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 181)
    scores = []
    for t in grid:
        scores.append(matthews_corrcoef(y_true, (proba >= t).astype(int)))
    best_i = int(np.argmax(scores))
    return float(grid[best_i]), float(scores[best_i])


def cv_oof_and_test_proba_lgb(X, y, X_test, seed, n_splits=5):
    params = dict(
        n_estimators=6000,
        learning_rate=0.015,
        num_leaves=63,
        min_child_samples=15,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=seed,
        n_jobs=-1,
    )

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof = np.zeros(len(X), dtype=float)
    test_proba = np.zeros(len(X_test), dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = LGBMClassifier(**params)

        # LightGBM can handle NaNs directly; no scaler needed.
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="binary_logloss",
        )

        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        test_proba += model.predict_proba(X_test)[:, 1] / n_splits

    return oof, test_proba


# =====================
# Load data
# =====================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

ID_COL = train.columns[0]  # unnamed ID column
TARGET_COL = "target_class"

train_ids = train[ID_COL].values
test_ids = test[ID_COL].values

X = train.drop(columns=[ID_COL, TARGET_COL])
y = train[TARGET_COL].astype(int)
X_test = test.drop(columns=[ID_COL])

# =====================
# Train 2 seeds
# =====================
oof1, testp1 = cv_oof_and_test_proba_lgb(
    X, y, X_test, seed=RANDOM_STATE_1, n_splits=N_SPLITS
)
t1, mcc1 = mcc_optimal_threshold(y, oof1)

oof2, testp2 = cv_oof_and_test_proba_lgb(
    X, y, X_test, seed=RANDOM_STATE_2, n_splits=N_SPLITS
)
t2, mcc2 = mcc_optimal_threshold(y, oof2)

# Seed-specific hard predictions
pred1 = (testp1 >= t1).astype(int)
pred2 = (testp2 >= t2).astype(int)

# Majority vote for 2 models (handle ties using avg-proba threshold optimized on avg OOF)
oof_avg = (oof1 + oof2) / 2.0
t_avg, mcc_avg = mcc_optimal_threshold(y, oof_avg)

testp_avg = (testp1 + testp2) / 2.0
pred_tie = (testp_avg >= t_avg).astype(int)

# If both agree → keep; if disagree → use tie-breaker
final_pred = np.where(pred1 == pred2, pred1, pred_tie)

print(f"[Seed {RANDOM_STATE_1}] Best OOF MCC={mcc1:.5f} @ t={t1:.3f}")
print(f"[Seed {RANDOM_STATE_2}] Best OOF MCC={mcc2:.5f} @ t={t2:.3f}")
print(f"[Avg OOF] Best OOF MCC={mcc_avg:.5f} @ t={t_avg:.3f}")

sub_dual = pd.DataFrame({"ID": test_ids, "target": final_pred})
sub_dual.to_csv("results_dual_seed.csv", index=False)
print("Saved: results_dual_seed.csv")

[LightGBM] [Info] Number of positive: 15360, number of negative: 3840
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000800 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5941
[LightGBM] [Info] Number of data points in the train set: 19200, number of used features: 42
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.800000 -> initscore=1.386294
[LightGBM] [Info] Start training from score 1.386294
[LightGBM] [Info] Number of positive: 15360, number of negative: 3840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000698 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5916
[LightGBM] [Info] Number of data points in the train set: 19200, number of used features: 42
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.800000 -> initscore=1.386294
[LightG